# AI013 — Time Series Forecasting: Training Notebook

**Task:** AI013 - Time Series Forecasting  
**Author:** Divjot  
**Branch:** ai-ml/ai013/divjot-forecasting  

This notebook covers:
1. Data loading and preparation
2. Sequence creation
3. Model training
4. Saving the trained model

## 1. Setup & Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Add forecasting module to path
sys.path.append(str(Path('../../src/model/forecasting').resolve()))

from dataset_builder import build_dataset
from sequence_generator import create_sequences, train_test_split_sequences
from model import ForecastingModel
from trainer import train

print('All modules loaded successfully.')

## 2. Load and Prepare Data

In [ ]:
# Path to cleaned data from AI003 pipeline
DATA_PATH = Path('../../cleaning/data/output/cleaned_data.csv').resolve()

print(f'Loading data from: {DATA_PATH}')
df = build_dataset(str(DATA_PATH))

print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Check data types and missing values
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

## 3. Visualise the Time Series

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['timestamp'], df['severity_score'], marker='o', linewidth=1, markersize=4)
plt.title('Severity Score Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Severity Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['timestamp'], df['risk_level_encoded'], marker='s', linewidth=1, markersize=4, color='orange')
plt.title('Risk Level (Encoded) Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Risk Level (0=Unknown, 1=Low, 2=Medium, 3=High, 4=Severe, 5=Critical)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Create Sequences

In [ ]:
SEQUENCE_LENGTH = 5  # Use 5 past records to predict next

X, y = create_sequences(df, target_col='severity_score', sequence_length=SEQUENCE_LENGTH)

print(f'Total sequences: {len(X)}')
print(f'X shape: {X.shape}  (samples, sequence_length, features)')
print(f'y shape: {y.shape}  (samples,)')
print(f'\nExample sequence input:\n{X[0]}')
print(f'\nCorresponding target: {y[0]}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split_sequences(X, y, test_ratio=0.2)

print(f'Training samples:  {len(X_train)}')
print(f'Testing samples:   {len(X_test)}')

## 5. Build and Train the Model

In [ ]:
INPUT_SIZE = X_train.shape[2]   # number of features
HIDDEN_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 0.001
BATCH_SIZE = 4

print(f'Model config:')
print(f'  Input size:    {INPUT_SIZE}')
print(f'  Hidden size:   {HIDDEN_SIZE}')
print(f'  Epochs:        {EPOCHS}')
print(f'  Learning rate: {LEARNING_RATE}')

model = ForecastingModel(input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE)
print('\nModel created.')

In [ ]:
print('Starting training...')
loss_history = train(
    model,
    X_train, y_train,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE
)
print('\nTraining complete!')

## 6. Plot Training Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(loss_history)+1), loss_history, linewidth=2, color='blue')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'Final training loss: {loss_history[-1]:.4f}')

## 7. Save the Trained Model

In [ ]:
MODEL_SAVE_PATH = Path('../../models/ai013_forecasting/forecasting_model.npz').resolve()
MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

model.save(str(MODEL_SAVE_PATH))
print(f'Model saved to: {MODEL_SAVE_PATH}')

## 8. Summary

In [ ]:
print('=== Training Summary ===')
print(f'Dataset records:     {len(df)}')
print(f'Sequence length:     {SEQUENCE_LENGTH}')
print(f'Training sequences:  {len(X_train)}')
print(f'Testing sequences:   {len(X_test)}')
print(f'Epochs trained:      {EPOCHS}')
print(f'Final loss:          {loss_history[-1]:.4f}')
print(f'Model saved:         {MODEL_SAVE_PATH}')